# PI3Kα (4JPS) — Step 5B: Receptor side-chain repair and Meeko retry

**Input:** `PI3Kalpha_4JPS_Step5.zip`  
**Purpose:** Repair missing atoms in existing residues using PDBFixer, record modifications, check original pocket coordinates are preserved, retry Meeko. This does **not** validate docking. Keep both chains A+B; do not rebuild missing sequence segments automatically. Confirm hydrogen/protonation choices and receptor suitability before redocking.

**Note:** PDBFixer may choose one altloc conformer when it reads the original PDB. Record that choice as a modelling limitation. If repair or PDBQT conversion fails, download diagnostics and review rather than deleting residues.

In [1]:
!pip -q install biopython numpy pandas meeko rdkit openmm
!pip -q install git+https://github.com/openmm/pdbfixer.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import files
from pathlib import Path
import zipfile, json, subprocess, sys, shutil, traceback
import numpy as np
from Bio.PDB import PDBParser

out = Path('PI3Kalpha_4JPS_Step5B'); out.mkdir(exist_ok=True)
print('Upload PI3Kalpha_4JPS_Step5.zip')
uploaded = files.upload()
zs = [k for k in uploaded if k.lower().endswith('.zip')]
if len(zs) != 1: raise RuntimeError(f'Upload exactly one ZIP; found {zs}')
needed = ['4JPS_protein_AB_preliminary.pdb','1LT_crystal_heavy_atom_mapped.sdf','1LT_reference.pdbqt']
with zipfile.ZipFile(zs[0]) as z:
    if z.testzip(): raise RuntimeError('ZIP integrity failed')
    for name in needed:
        hits=[x for x in z.namelist() if x.split('/')[-1] == name]
        if len(hits) != 1: raise RuntimeError(f'Expected one {name}, found {len(hits)}')
        (out/name).write_bytes(z.read(hits[0]))
print('Extracted',needed)

Upload PI3Kalpha_4JPS_Step5.zip


Saving PI3Kalpha_4JPS_Step5.zip to PI3Kalpha_4JPS_Step5.zip
Extracted ['4JPS_protein_AB_preliminary.pdb', '1LT_crystal_heavy_atom_mapped.sdf', '1LT_reference.pdbqt']


In [5]:
# Repair missing ATOMS in existing residues; deliberately do NOT add absent sequence residues.
from pdbfixer import PDBFixer
from openmm.app import PDBFile
source = out/'4JPS_protein_AB_preliminary.pdb'
fixer = PDBFixer(filename=str(source))
fixer.findMissingResidues()
missing_sequence = {str(k): list(v) for k,v in fixer.missingResidues.items()}
fixer.missingResidues = {}  # prevents inserting speculative loops/residues
fixer.findNonstandardResidues()
nonstandard = [(r.chain.id, r.id, r.name, replacement) for r,replacement in fixer.nonstandardResidues]
if nonstandard: raise RuntimeError(f'Unexpected nonstandard protein residues: {nonstandard}')
fixer.findMissingAtoms()
missing_atoms = [{'chain':r.chain.id,'residue':r.id,'name':r.name,'atoms':[a.name for a in atoms]}
                 for r,atoms in fixer.missingAtoms.items()]

missing_terminal = [
    {
        'chain': r.chain.id,
        'residue': r.id,
        'name': r.name,
        'atoms': [a if isinstance(a, str) else a.name for a in atoms]
    }
    for r, atoms in fixer.missingTerminals.items()
]

print('Residues with missing sidechain/backbone atoms:',len(missing_atoms))
print('Missing terminal atom groups:',len(missing_terminal))
print('Example repairs:', missing_atoms[:12])
fixer.addMissingAtoms()
# Explicitly record the pH assumed when choosing protonation variants.
fixer.addMissingHydrogens(7.4)
repaired = out/'4JPS_AB_repaired_pH7p4.pdb'
with repaired.open('w') as f:
    PDBFile.writeFile(fixer.topology,fixer.positions,f,keepIds=True)
repair_log = {'input':str(source.name), 'output':str(repaired.name),
              'unbuilt_missing_sequence_segments':missing_sequence,
              'missing_atom_repairs':missing_atoms,'missing_terminal_repairs':missing_terminal,
              'assumed_pH':7.4, 'note':'PDBFixer rebuilt atoms and added H; altloc choices and protonation must be reviewed'}
(out/'repair_log.json').write_text(json.dumps(repair_log,indent=2)+'\n')
print('Saved repaired PDB:',repaired)

Residues with missing sidechain/backbone atoms: 45
Missing terminal atom groups: 2
Example repairs: [{'chain': 'A', 'residue': '130', 'name': 'MET', 'atoms': ['CG', 'SD', 'CE']}, {'chain': 'A', 'residue': '132', 'name': 'LYS', 'atoms': ['CG', 'CD', 'CE', 'NZ']}, {'chain': 'A', 'residue': '179', 'name': 'LYS', 'atoms': ['CG', 'CD', 'CE', 'NZ']}, {'chain': 'A', 'residue': '187', 'name': 'LYS', 'atoms': ['CG', 'CD', 'CE', 'NZ']}, {'chain': 'A', 'residue': '204', 'name': 'LYS', 'atoms': ['CG', 'CD', 'CE', 'NZ']}, {'chain': 'A', 'residue': '244', 'name': 'LEU', 'atoms': ['CG', 'CD1', 'CD2']}, {'chain': 'A', 'residue': '245', 'name': 'GLU', 'atoms': ['CG', 'CD', 'OE1', 'OE2']}, {'chain': 'A', 'residue': '246', 'name': 'TYR', 'atoms': ['CG', 'CD1', 'CE1', 'CZ', 'OH', 'CE2', 'CD2']}, {'chain': 'A', 'residue': '247', 'name': 'GLN', 'atoms': ['CG', 'CD', 'OE1', 'NE2']}, {'chain': 'A', 'residue': '249', 'name': 'LYS', 'atoms': ['CG', 'CD', 'CE', 'NZ']}, {'chain': 'A', 'residue': '299', 'name': 'M

In [6]:
# Ensure all ORIGINAL protein atoms <= 5 Å from ligand retain their original coordinates.
from rdkit import Chem
lig=next(m for m in Chem.SDMolSupplier(str(out/'1LT_crystal_heavy_atom_mapped.sdf')) if m is not None)
lig_xyz=np.asarray(lig.GetConformer().GetPositions())
p=PDBParser(QUIET=True)
original=p.get_structure('orig',str(source))
fixed=p.get_structure('fixed',str(repaired))
checks=[]
missing_pocket=[]
for chain in original[0]:
    for res in chain:
        if res.id[0] != ' ': continue
        for atom in res:
            if atom.element in ('H','D'):continue
            d=float(np.linalg.norm(lig_xyz-np.asarray(atom.coord),axis=1).min())
            if d > 5.0:continue
            key=(chain.id,res.id,atom.name)
            try:
                atom2=fixed[0][chain.id][res.id][atom.name]
                checks.append(float(np.linalg.norm(atom.coord - atom2.coord)))
            except KeyError:
                missing_pocket.append(str(key))
max_deviation=max(checks) if checks else float('inf')
print('Original pocket heavy atoms checked:',len(checks))
print('Original pocket heavy atoms missing:',len(missing_pocket))
print('Maximum displacement (Å):',max_deviation)
repair_log['pocket_original_heavy_atoms_checked']=len(checks)
repair_log['pocket_original_heavy_atoms_missing']=missing_pocket
repair_log['max_original_pocket_heavy_atom_displacement_A']=max_deviation
(out/'repair_log.json').write_text(json.dumps(repair_log,indent=2)+'\n')
if missing_pocket or max_deviation > 0.15:
    raise RuntimeError('Original pocket coordinates changed/missing: stop and investigate')
print('PASS: original local-pocket heavy-atom coordinates retained')

Original pocket heavy atoms checked: 85
Original pocket heavy atoms missing: 0
Maximum displacement (Å): 0.0
PASS: original local-pocket heavy-atom coordinates retained


In [9]:

%pip install -q gemmi

import gemmi
print("Gemmi installed:", gemmi.__version__)


Gemmi installed: 0.7.5


In [10]:
# Re-run Meeko. Even success here is provisional; redocking validation comes next.
cmd=[sys.executable,'-m','meeko.cli.mk_prepare_receptor','--read_pdb',str(repaired),
     '--default_altloc','A', '-o',str(out/'4JPS_AB_repaired'),'-p']
result=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
(out/'receptor_meeko_retry.log').write_text('COMMAND: '+' '.join(cmd)+'\n'+result.stdout+'\nRETURN CODE: '+str(result.returncode))
pdbqts=[x for x in out.glob('4JPS_AB_repaired*.pdbqt') if x.stat().st_size>0]
print('Meeko exit status:',result.returncode)
print(result.stdout[-4000:])
print('Receptor PDBQT:',[x.name for x in pdbqts])
print('STATUS:', 'PROVISIONAL receptor conversion successful, not docking validated' if result.returncode==0 and pdbqts else 'Not prepared; inspect log')
(out/'step5b_status.json').write_text(json.dumps({'meeko_exit_code':result.returncode,
   'receptor_pdbqt_prepared':bool(result.returncode==0 and pdbqts),
   'reference_redocking_completed':False,
   'scientific_status':'provisional; review alternate locations, protonation and repaired atoms'},indent=2)+'\n')

Meeko exit status: 1
[RDKit] ERROR:[08:05:17] Explicit valence for atom # 13 C, 5, is greater than permitted
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/meeko/polymer.py", line 2534, in _pdb_to_residue_mols
    pdbmol, _, missed_altloc, needed_altloc = _aux_altloc_mol_build(
                                              ~~~~~~~~~~~~~~~~~~~~~^
        atom_field_list,
        ^^^^^^^^^^^^^^^^
        requested_altloc,
        ^^^^^^^^^^^^^^^^^
        default_altloc,
        ^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/meeko/utils/rdkitutils.py", line 415, in _aux_altloc_mol_build
    _ = Chem.SanitizeMol(pdbmol)
rdkit.Chem.rdchem.AtomValenceException: Explicit valence for atom # 13 C, 5, is greater than permitted

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_co

203

In [11]:

from pathlib import Path
import subprocess
import sys

out = Path("PI3Kalpha_4JPS_Step5B")

source = out / "4JPS_AB_repaired_pH7p4.pdb"
heavy_only = out / "4JPS_AB_repaired_heavy_only.pdb"

if not source.exists():
    raise FileNotFoundError(f"Missing: {source}")

# Remove generated hydrogen atoms only.
lines = source.read_text().splitlines()
cleaned = []

for line in lines:
    if line.startswith(("ATOM  ", "HETATM")):
        element = line[76:78].strip().upper()
        if element in {"H", "D"}:
            continue
    cleaned.append(line)

heavy_only.write_text(
    "\n".join(cleaned) + "\n"
)

print("Saved:", heavy_only)
print("Heavy-atom records:",
      sum(x.startswith(("ATOM  ", "HETATM"))
          for x in cleaned))

# Retry receptor preparation with Meeko.
cmd = [
    sys.executable,
    "-m",
    "meeko.cli.mk_prepare_receptor",
    "--read_pdb", str(heavy_only),
    "-o", str(out / "4JPS_AB_heavy_retry"),
    "-p",
    "--default_altloc", "A"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

log = result.stdout + "\n" + result.stderr
(out / "receptor_heavy_retry.log").write_text(log)

print("Meeko exit status:", result.returncode)
print(log[-6000:])

pdbqt = out / "4JPS_AB_heavy_retry.pdbqt"
print(
    "Receptor PDBQT generated:",
    result.returncode == 0
    and pdbqt.exists()
    and pdbqt.stat().st_size > 0
)


Saved: PI3Kalpha_4JPS_Step5B/4JPS_AB_repaired_heavy_only.pdb
Heavy-atom records: 10133
Meeko exit status: 1
Template padding will be used.
 gasteiger charges will be read from template file 
 

Error: Creation of data structure for receptor failed.

Details:
- Template matching failed for: ['A:245', 'A:246', 'A:298', 'A:479', 'A:481', 'A:697', 'B:435', 'B:438', 'B:509', 'B:513']
These residues can be ignored with option --delete_bad_res or --delete_bad_res_from_box_radius.

Recommendations:
1. (for batch processing) Use -a/--allow_bad_res to automatically remove residues
that do not match templates, and --default_altloc to set
a default altloc variant. Use these at your own risk.

2. (processing individual structure) Inspecting and fixing the input structure is recommended.
Use --wanted_altloc to set variants for specific residues.



No template matched for residue_key='A:245'
tried 6 templates for residue_key='A:245'excess_H_ok=False
GLU        heavy_miss=0 heavy_excess=0 H_excess=[]

In [12]:

from Bio.PDB import PDBParser
import numpy as np
import pandas as pd
from pathlib import Path

pdb_file = Path(
    "PI3Kalpha_4JPS_Step5B/4JPS_AB_repaired_heavy_only.pdb"
)

structure = PDBParser(QUIET=True).get_structure(
    "PI3Kalpha", str(pdb_file)
)

problem_residues = {
    "A": [245, 246, 298, 479, 481, 697],
    "B": [435, 438, 509, 513]
}

center = np.array([-1.319, -9.513, 16.948])
box_size = np.array([18.946, 19.659, 25.114])
half_size = box_size / 2

results = []

for chain in structure[0]:
    if chain.id not in problem_residues:
        continue

    for residue in chain:
        if residue.id[1] not in problem_residues[chain.id]:
            continue

        coords = np.array([
            atom.coord for atom in residue
            if atom.element.upper() not in {"H", "D"}
        ])

        if len(coords) == 0:
            continue

        distances = np.linalg.norm(coords - center, axis=1)

        # Minimum distance from residue atoms to the
        # rectangular provisional docking box.
        outside = np.maximum(
            np.abs(coords - center) - half_size, 0
        )
        box_distances = np.linalg.norm(outside, axis=1)

        results.append({
            "Chain": chain.id,
            "Residue": residue.resname + str(residue.id[1]),
            "Min distance to box centre (A)":
                round(float(distances.min()), 2),
            "Min distance to box surface (A)":
                round(float(box_distances.min()), 2),
            "Within provisional box":
                bool(np.any(box_distances == 0))
        })

df = pd.DataFrame(results)
display(df.sort_values(
    "Min distance to box surface (A)"
))

df.to_csv(
    "PI3Kalpha_4JPS_Step5B/"
    "problem_residue_distances.csv",
    index=False
)


,Chain,Residue,Min distance to box centre (A),Min distance to box surface (A),Within provisional box
5,A,MET697,31.75,20.58,False
2,A,PRO298,35.72,24.87,False
1,A,TYR246,44.38,29.95,False
7,B,LYS438,50.12,32.33,False
6,B,GLN435,49.71,32.96,False
0,A,GLU245,47.79,33.49,False
3,A,TRP479,61.45,43.28,False
4,A,SER481,65.53,47.40,False
8,B,ILE509,67.24,48.96,False
9,B,LYS513,71.11,52.87,False


In [13]:

from pathlib import Path
import subprocess
import sys

out = Path("PI3Kalpha_4JPS_Step5B")

input_pdb = out / "4JPS_AB_repaired_heavy_only.pdb"
output_base = out / "4JPS_AB_distance_filtered"

assert input_pdb.exists(), "Repaired receptor PDB not found"

cmd = [
    sys.executable,
    "-m", "meeko.cli.mk_prepare_receptor",
    "--read_pdb", str(input_pdb),
    "-o", str(output_base),
    "-p",
    "--default_altloc", "A",
    "--box_center",
    "-1.319", "-9.513", "16.948",
    "--box_size",
    "18.946", "19.659", "25.114",
    "--delete_bad_res_from_box_radius", "10"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

log = result.stdout + "\n" + result.stderr
(out / "receptor_distance_filtered.log").write_text(log)

print("Meeko exit status:", result.returncode)
print(log[-7000:])

pdbqt = Path(str(output_base) + ".pdbqt")

success = (
    result.returncode == 0
    and pdbqt.exists()
    and pdbqt.stat().st_size > 0
)

print("Receptor PDBQT generated:", success)

if success:
    print("File:", pdbqt)
    print("Size:", pdbqt.stat().st_size, "bytes")
    print("NEXT: inspect excluded residues and pocket integrity.")
else:
    print("STOP: review the new Meeko log.")


Meeko exit status: 0
Template padding will be used.
 gasteiger charges will be read from template file 
 

Files written:
PI3Kalpha_4JPS_Step5B/4JPS_AB_distance_filtered.pdbqt <-- static (i.e., rigid) receptor input file

No template matched for residue_key='A:245'
tried 6 templates for residue_key='A:245'excess_H_ok=False
GLU        heavy_miss=0 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={14}
NGLU       heavy_miss=0 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={16, 4}
CGLU       heavy_miss=1 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={1, 15}
GLH        heavy_miss=0 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={14}
NGLH       heavy_miss=0 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={16, 4}
CGLH       heavy_miss=1 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={1, 15}

matched with excess inter-residue bond(s): A:245
No template matched for residue_key='A:246'
tried 3 templates for residue_key='A:246'excess_H_ok=False
TYR   

In [16]:

from pathlib import Path
import requests

out = Path("PI3Kalpha_4JPS_Step5B")
out.mkdir(exist_ok=True)

original = out / "4JPS_original.pdb"

if not original.exists():
    url = "https://files.rcsb.org/download/4JPS.pdb"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    original.write_bytes(response.content)
    print("Downloaded original 4JPS PDB")
else:
    print("Original PDB already exists")

prepared = out / "4JPS_AB_distance_filtered.pdbqt"

print("Original PDB exists:", original.exists())
print("Prepared receptor exists:", prepared.exists())



Downloaded original 4JPS PDB
Original PDB exists: True
Prepared receptor exists: True


In [17]:

from pathlib import Path
from Bio.PDB import PDBParser
import numpy as np

out = Path("PI3Kalpha_4JPS_Step5B")

original = out / "4JPS_original.pdb"
prepared = out / "4JPS_AB_distance_filtered.pdbqt"

assert original.exists(), "Original PDB missing"
assert prepared.exists(), "Prepared PDBQT missing"

structure = PDBParser(QUIET=True).get_structure(
    "original", str(original)
)

# Locate experimentally bound 1LT
ligand = [
    r for chain in structure[0] for r in chain
    if r.resname == "1LT"
][0]

lig_xyz = np.array([
    a.coord for a in ligand
    if a.element.upper() not in {"H", "D"}
])

# Read prepared receptor heavy atoms by residue/atom identity
prepared_atoms = {}

for line in prepared.read_text().splitlines():
    if not line.startswith(("ATOM  ", "HETATM")):
        continue

    name = line[12:16].strip()
    chain = line[21].strip()
    resnum = int(line[22:26])
    xyz = np.array([
        float(line[30:38]),
        float(line[38:46]),
        float(line[46:54])
    ])

    atom_type = line.split()[-1].upper()
    if atom_type in {"H", "HD"}:
        continue

    prepared_atoms[(chain, resnum, name)] = xyz

# Compare every original protein heavy atom within 5 Å
nearby = []
missing = []
displacements = []

for chain in structure[0]:
    if chain.id not in {"A", "B"}:
        continue

    for residue in chain:
        if residue.id[0] != " ":
            continue

        for atom in residue:
            if atom.element.upper() in {"H", "D"}:
                continue

            distance = float(
                np.linalg.norm(lig_xyz - atom.coord, axis=1).min()
            )

            if distance > 5.0:
                continue

            key = (chain.id, residue.id[1], atom.name)
            nearby.append(key)

            if key not in prepared_atoms:
                missing.append(key)
            else:
                displacements.append(
                    np.linalg.norm(prepared_atoms[key] - atom.coord)
                )

print("Original pocket heavy atoms:", len(nearby))
print("Missing in final PDBQT:", len(missing))
print("Maximum coordinate displacement (Å):",
      round(max(displacements), 4) if displacements else "N/A")
print("Missing atom identities:", missing[:20])

if not missing and displacements and max(displacements) < 0.1:
    print("PASS: Original 5 Å binding-pocket heavy atoms preserved")
else:
    print("REVIEW REQUIRED: Do not begin redocking yet")


Original pocket heavy atoms: 85
Missing in final PDBQT: 0
Maximum coordinate displacement (Å): 0.0
Missing atom identities: []
PASS: Original 5 Å binding-pocket heavy atoms preserved


In [18]:

from pathlib import Path
from google.colab import files
import zipfile

out = Path("PI3Kalpha_4JPS_Step5B")

required = [
    "4JPS_AB_distance_filtered.pdbqt",
    "4JPS_original.pdb",
    "4JPS_AB_repaired_heavy_only.pdb",
    "receptor_distance_filtered.log"
]

package = "PI3Kalpha_4JPS_Prepared_Receptor.zip"

with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as z:
    for name in required:
        path = out / name
        if not path.exists():
            raise FileNotFoundError(f"Missing: {path}")
        z.write(path, arcname=name)

print("Prepared receptor package saved:", package)
files.download(package)


Prepared receptor package saved: PI3Kalpha_4JPS_Prepared_Receptor.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Download diagnostics even if Meeko failed.
archive=shutil.make_archive('PI3Kalpha_4JPS_Step5B','zip',out)
print('Downloading',archive)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>